# Module 1: Vector Search Fundamentals

This notebook implements a semantic search system for Airbnb listings using:
- **OpenAI Embeddings** (text-embedding-3-small) for vector generation
- **DocumentDB** with cosmosSearch for vector similarity search
- **Filters** for refining search results

## Learning Objectives
- Generate vector embeddings using OpenAI
- Create vector search indexes in DocumentDB
- Implement semantic search with similarity scoring
- Apply filters to refine search results

## Step 1: Import Required Libraries and Setup Environment

In [ ]:
import os
import json
from pymongo import MongoClient
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv(override=True)

# Initialize OpenAI client
openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✅ Libraries imported and environment loaded")

## Step 2: Connect to DocumentDB

In [ ]:
# Connect to DocumentDB
DOCUMENTDB_CONNECTION_STRING = os.getenv('DOCUMENTDB_CONNECTION_STRING')
client = MongoClient(DOCUMENTDB_CONNECTION_STRING)
db = client['db']
collection = db['listings']

print("✅ Connected to DocumentDB")
print(f"📊 Current document count: {collection.count_documents({})}")

In [ ]:
# Verify connection by fetching one document
test_doc = collection.find_one()
if test_doc:
    print(f"✅ Successfully retrieved document: {test_doc.get('name', 'Unknown')}")
else:
    print("⚠️ No documents found. We'll load data next.")

## Step 3: Load and Examine Sample Data

Let's explore the dataset structure to understand what fields are available.

In [ ]:
# Load a small sample to examine
with open('data/raw_data.json', 'r') as f:
    data = json.load(f)

# Look at the first listing
sample = data[0]
print(f"Listing ID: {sample['id']}")
print(f"Name: {sample['name']}")
print(f"Property Type: {sample['property_type']}")
print(f"Bedrooms: {sample.get('bedrooms', 'N/A')}")
print(f"Price: ${sample.get('price', 'N/A')}")
print(f"Amenities: {', '.join(sample.get('amenities', [])[:5])}...")
print(f"\nDescription Preview:")
print(sample.get('description', '')[:200] + "...")

## Step 4: Create Embedding Generation Function

We'll use OpenAI's `text-embedding-3-small` model to generate 1536-dimension vectors that capture semantic meaning.

In [ ]:
def generate_embedding(text):
    """
    Generate a vector embedding for the given text using OpenAI.
    
    Args:
        text (str): The text to embed
        
    Returns:
        list: A 1536-dimension vector representing the text
    """
    if not text or not isinstance(text, str):
        return None
    
    try:
        response = openai_client.embeddings.create(
            model="text-embedding-3-small",
            input=text
        )
        return response.data[0].embedding
    except Exception as e:
        print(f"Error generating embedding: {e}")
        return None

# Test the function
test_text = "Cozy apartment near downtown with free parking"
test_embedding = generate_embedding(test_text)

print(f"✅ Generated embedding")
print(f"📏 Dimensions: {len(test_embedding)}")
print(f"📊 First 5 values: {test_embedding[:5]}")
print(f"📊 Data type: {type(test_embedding[0])}")

## Step 5: Load Data with Embeddings

This function loads JSON data and generates embeddings for each listing's description field.

In [ ]:
def load_data_with_embeddings(file_path, limit=None):
    """
    Load data from JSON file and generate embeddings for each listing.
    
    Args:
        file_path (str): Path to the JSON data file
        limit (int, optional): Maximum number of documents to process
        
    Returns:
        list: Documents with embeddings added
    """
    print(f"📖 Loading data from {file_path}...")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    if limit:
        data = data[:limit]
    
    print(f"📊 Loaded {len(data)} documents")
    print("🔄 Generating embeddings...")
    
    documents_with_embeddings = []
    
    for idx, doc in enumerate(data):
        # Create a rich description for embedding
        description_text = doc.get('description', '')
        
        # Generate embedding
        embedding = generate_embedding(description_text)
        
        if embedding:
            doc['descriptionVector'] = embedding
            documents_with_embeddings.append(doc)
            
            if (idx + 1) % 10 == 0:
                print(f"  Processed {idx + 1}/{len(data)} documents...")
    
    print(f"✅ Generated embeddings for {len(documents_with_embeddings)} documents")
    return documents_with_embeddings

In [ ]:
# Start with a small dataset for testing (50 documents)
documents = load_data_with_embeddings(
    'data/raw_data.json',
    limit=50
)

In [ ]:
# Show one document from the documents list
print(f"📄 Sample document with new embeddings:\n")
sample_doc = documents[0]
print(f"ID: {sample_doc.get('id')}")
print(f"Name: {sample_doc.get('name')}")
print(f"Property Type: {sample_doc.get('property_type')}")
print(f"Bedrooms: {sample_doc.get('bedrooms', 'N/A')}")
print(f"Price: ${sample_doc.get('price', 'N/A')}")
print(f"Has embedding: {'descriptionVector' in sample_doc}")
print(f"Embedding dimensions: {len(sample_doc.get('descriptionVector', []))}")

# Step 6: Create Vector Index Using the DocumentDB for VS Code Extension

Now that your data with embeddings is loaded in DocumentDB, you need to create a **vector search index** to enable fast similarity searches.

1. **Open the DocumentDB Extension** in VS Code (click the database icon in the sidebar)

2. **Navigate to your Scrapbook**:
   - Right-click on your connection
   - Select **"New Scrapbook"** (or open an existing `.mongodb` scrapbook file)

3. **Run the following commands** in your scrapbook (select each block and press `Ctrl+Enter` or click "Run"):

In [ ]:
// Create vector search index on the descriptionVector field
db.runCommand({
    createIndexes: "listings",
    indexes: [{
        key: { "descriptionVector": "cosmosSearch" },
        name: "vectorSearchIndex",
        cosmosSearchOptions: {
            kind: "vector-ivf",
            numLists: 100,
            similarity: "COS",
            dimensions: 1536
        }
    }]
})

// Create filter indexes
db.listings.createIndex({ "address.market": 1 })
db.listings.createIndex({ "property_type": 1 })
db.listings.createIndex({ "bedrooms": 1 })
db.listings.createIndex({ "price": 1 })

// Check all indexes on the collection
db.listings.getIndexes()

## Step 7: Create Vector Search Index

DocumentDB supports native vector search using the `cosmosSearch` operator. We'll use **IVF (Inverted File Index)** for efficient approximate search.

### Index Parameters:
- **kind**: `vector-ivf` for speed
- **numLists**: Controls speed/accuracy tradeoff
- **similarity**: `COS` for cosine similarity (range: -1 to 1)
- **dimensions**: 1536 (must match embedding size)

In [ ]:
def create_vector_index():
    """
    Create a vector search index on the descriptionVector field.
    Uses IVF (Inverted File Index) for efficient approximate search.
    """
    try:
        # Drop existing vector index if it exists
        try:
            collection.drop_index("vectorSearchIndex")
            print("🗑️ Dropped existing vector index")
        except:
            pass  # Index doesn't exist yet
        
        # Create vector search index
        collection.create_index(
            [("descriptionVector", "cosmosSearch")],
            name="vectorSearchIndex",
            cosmosSearchOptions={
                "kind": "vector-ivf",
                "numLists": 100,  # Number of clusters for IVF
                "similarity": "COS",  # Cosine similarity
                "dimensions": 1536  # Must match embedding dimensions
            }
        )
        print("✅ Created vector search index (IVF)")
        
        # Also create indexes for filtering
        collection.create_index([("address.market", 1)])
        collection.create_index([("property_type", 1)])
        collection.create_index([("bedrooms", 1)])
        collection.create_index([("price", 1)])
        print("✅ Created filter indexes")
        
    except Exception as e:
        print(f"❌ Error creating indexes: {e}")
        raise

# Create the indexes
create_vector_index()

# Verify indexes
indexes = list(collection.list_indexes())
print(f"\n📋 Current indexes:")
for idx in indexes:
    print(f"  - {idx['name']}: {idx['key']}")

## Step 8: Implement Basic Semantic Search

Now we can search for listings using natural language! The search converts the query to an embedding and finds the most similar listings.

### Understanding Search Scores:
- Scores range from 0 to 1 (with cosine similarity)
- **> 0.75**: Strong semantic relevance
- **0.5 - 0.75**: Moderately relevant
- **< 0.5**: Weak matches

In [ ]:
def search_listings(query, limit=5):
    """
    Search for listings using semantic similarity.
    
    Args:
        query (str): Natural language search query
        limit (int): Maximum number of results to return
        
    Returns:
        list: Matching listings with similarity scores
    """
    # Generate embedding for the query
    query_embedding = generate_embedding(query)
    
    if not query_embedding:
        print("❌ Failed to generate query embedding")
        return []
    
    # Perform vector search using cosmosSearch
    pipeline = [
        {
            "$search": {
                "cosmosSearch": {
                    "vector": query_embedding,
                    "path": "descriptionVector",
                    "k": limit  # Number of nearest neighbors
                },
                "returnStoredSource": True
            }
        },
        {
            "$project": {
                "_id": 1,
                "name": 1,
                "description": 1,
                "property_type": 1,
                "bedrooms": 1,
                "beds": 1,
                "price": 1,
                "address.market": 1,
                "amenities": 1,
                "searchScore": {"$meta": "searchScore"}
            }
        }
    ]
    
    results = list(collection.aggregate(pipeline))
    return results

In [ ]:
# Test the search
query = "cozy apartment with parking near downtown"
results = search_listings(query, limit=5)

print(f"\n🔍 Search Query: '{query}'")
print(f"📊 Found {len(results)} results\n")

for idx, result in enumerate(results, 1):
    print(f"{idx}. {result['name']}")
    print(f"   Property Type: {result.get('property_type', 'N/A')}")
    print(f"   Location: {result.get('address', {}).get('market', 'N/A')}")
    print(f"   Bedrooms: {result.get('bedrooms', 'N/A')} | Price: ${result.get('price', 'N/A')}")
    print(f"   Similarity Score: {result.get('searchScore', 0):.4f}")
    print(f"   Preview: {result.get('description', '')[:100]}...")
    print()

## Step 9: Search with Filters

Combine semantic search with traditional filters like bedrooms, price, market, and amenities for more refined results.

In [ ]:
def search_listings_with_filters(query, filters=None, limit=5):
    """
    Search for listings with semantic similarity and additional filters.
    
    Args:
        query (str): Natural language search query
        filters (dict): Optional filters (bedrooms, price_max, market, amenities)
        limit (int): Maximum number of results to return
        
    Returns:
        list: Matching listings with similarity scores
    """
    # Generate embedding for the query
    query_embedding = generate_embedding(query)
    
    if not query_embedding:
        print("❌ Failed to generate query embedding")
        return []
    
    # Build match stage for filters
    match_conditions = {}
    
    if filters:
        if 'bedrooms' in filters:
            match_conditions['bedrooms'] = {"$gte": filters['bedrooms']}
        
        if 'price_max' in filters:
            match_conditions['price'] = {"$lte": filters['price_max']}
        
        if 'market' in filters:
            match_conditions['address.market'] = filters['market']
        
        if 'amenities' in filters:
            # Amenities is a list, so we check if all required amenities are present
            match_conditions['amenities'] = {"$all": filters['amenities']}
    
    # Build aggregation pipeline
    pipeline = [
        {
            "$search": {
                "cosmosSearch": {
                    "vector": query_embedding,
                    "path": "descriptionVector",
                    "k": limit * 10  # Fetch more to account for filtering
                },
                "returnStoredSource": True
            }
        }
    ]
    
    # Add filter stage if we have conditions
    if match_conditions:
        pipeline.append({"$match": match_conditions})
    
    # Add projection and limit
    pipeline.extend([
        {
            "$project": {
                "_id": 1,
                "name": 1,
                "description": 1,
                "property_type": 1,
                "bedrooms": 1,
                "beds": 1,
                "price": 1,
                "address.market": 1,
                "amenities": 1,
                "searchScore": {"$meta": "searchScore"}
            }
        },
        {"$limit": limit}
    ])
    
    results = list(collection.aggregate(pipeline))
    return results

In [ ]:
# Test with filters
query = "family-friendly home with outdoor space"
filters = {
    "bedrooms": 3,
    "price_max": 200,
    "amenities": ["Wifi", "Kitchen"]
}

results = search_listings_with_filters(query, filters, limit=5)

print(f"\n🔍 Search Query: '{query}'")
print(f"🎯 Filters:")
print(f"   - Bedrooms: {filters['bedrooms']}+")
print(f"   - Max Price: ${filters['price_max']}")
print(f"   - Amenities: {', '.join(filters['amenities'])}")
print(f"\n📊 Found {len(results)} results\n")

for idx, result in enumerate(results, 1):
    print(f"{idx}. {result['name']}")
    print(f"   Property Type: {result.get('property_type', 'N/A')}")
    print(f"   Location: {result.get('address', {}).get('market', 'N/A')}")
    print(f"   Bedrooms: {result.get('bedrooms', 'N/A')} | Price: ${result.get('price', 'N/A')}")
    print(f"   Similarity Score: {result.get('searchScore', 0):.4f}")
    amenities_preview = ', '.join(result.get('amenities', [])[:5])
    print(f"   Amenities: {amenities_preview}...")
    print()

## Step 10: Experiment with Different Queries

Test the semantic search with various natural language queries to see how it understands context and intent.

**💡 Observations:**
- "romantic getaway" finds properties with ambiance descriptions
- "pet-friendly" matches listings mentioning pets, animals, or outdoor areas
- "business travel" finds properties with workspaces and good wifi
- The semantic understanding goes beyond exact keyword matching!

In [ ]:
# Test various semantic queries
test_queries = [
    "romantic getaway for couples",
    "pet-friendly place near parks",
    "business travel with home office",
    "beachfront property for surfing",
    "quiet retreat for meditation and yoga"
]

print("🧪 Testing Semantic Search Capabilities\n")
print("=" * 80)

for query in test_queries:
    results = search_listings(query, limit=3)
    
    print(f"\n🔍 Query: '{query}'")
    print(f"📊 Top 3 Results:")
    
    for idx, result in enumerate(results, 1):
        print(f"\n   {idx}. {result['name']}")
        print(f"      Score: {result.get('searchScore', 0):.4f}")
        print(f"      {result.get('property_type', 'N/A')} | "
              f"{result.get('bedrooms', 'N/A')} bed | "
              f"${result.get('price', 'N/A')}/night")
    
    print("\n" + "-" * 80)

## 🎓 What You've Learned

✅ **Vector Embeddings**: Converting text into numerical representations  
✅ **OpenAI Embeddings API**: Using text-embedding-3-small for semantic encoding  
✅ **DocumentDB Vector Indexes**: Creating IVF indexes for efficient similarity search  
✅ **Semantic Search**: Implementing cosine similarity search with cosmosSearch  
✅ **Search Filters**: Combining vector search with traditional filters  
✅ **Query Understanding**: How embeddings capture meaning and context

## 🎉 What's Next?

In **Module 2: RAG Pattern Implementation**, you'll learn how to:
- Build a conversational AI that uses your vector search
- Implement Retrieval-Augmented Generation (RAG) with LangChain
- Create context-aware responses using retrieved listings
- Handle conversation memory and follow-up questions